# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is a Croissant schema available at:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`.

We use `dataset.schema` to access detailed schema information including all defined record sets and fields. Throughout, we reference every Croissant entity by its `@id`.

In [ ]:
# Explore record sets and fields by @id
from pprint import pprint

schema = dataset.schema

print("Available record sets (@id, name):")
record_set_ids = []
for rs in schema.get('recordSet', []):
    rid = rs.get('@id', None)
    rname = rs.get('name', '')
    print(f"  - {rid}: {rname}")
    record_set_ids.append(rid)

if not record_set_ids:
    print("No record sets defined in 'recordSet' array. Loading default records if available.")

# For each record set, print fields and their @id
for rs in schema.get('recordSet', []):
    print(f"\nFields for record set {rs.get('@id')}: {rs.get('name','')}")
    fields = rs.get('field', [])
    # Each field can be a dict or an @id ref
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            print(f"  - {field.get('@id','')}: {field.get('name','')}")
        else:
            print(f"  - {field}")

# If recordSet is empty, inform the user
if not schema.get('recordSet', []):
    print("Note: No explicit record sets in schema; data extraction will attempt to infer available structures.")

## 3. Data Extraction
Load data from the main record set(s) into a DataFrame, referencing them by their `@id`. You may need to refer to the overview above to confirm the available record sets and fields.

In [ ]:
# Extract record set IDs
record_sets = [rs.get('@id') for rs in schema.get('recordSet', [])]

dataframes = {}

if record_sets:
    for record_set_id in record_sets:
        # Extract by @id
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)

    main_rs = record_sets[0]
    print(f"Columns in main record set {main_rs}:")
    print(dataframes[main_rs].columns.tolist())
    display(dataframes[main_rs].head())
else:
    # No record sets explicitly defined: Try to load using default loader
    print("No explicit record sets; attempting to load all CSV/TSV distributions as default record sets...")
    from mlcroissant.tools.utils import download_schema
    from urllib.parse import urlparse
    import os

    # Download and parse schema to fetch distributions
    schema_json = download_schema(url)

    # Gather distributions by @id
    distributions = schema_json.get('distribution', [])
    if isinstance(distributions, dict):
        distributions = [distributions]
    for d in distributions:
        dist_id = d.get('@id', None) if isinstance(d, dict) else d
        print(f"  Distribution: {dist_id}")
        try:
            records = list(dataset.records(record_set=dist_id))
            df = pd.DataFrame(records)
            dataframes[dist_id] = df
            if not df.empty:
                print(f"Columns for distribution {dist_id}: {df.columns.tolist()}")
                display(df.head())
        except Exception as e:
            print(f"Could not load records for {dist_id}: {e}")

    if not dataframes:
        print("No record sets nor CSV-like distributions loaded.\nCheck the Croissant schema and documentation.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All field references use their `@id`.


In [ ]:
# Use main record set for EDA
if dataframes:
    main_rs = list(dataframes.keys())[0]
    df = dataframes[main_rs].copy()
    print(f"Analysing record set: {main_rs}")
    print("Columns:", df.columns.tolist())

    # Attempt to identify a likely numeric field (e.g., Age or similar)
    likely_numeric_fields = [
        c for c in df.columns if any(x in c.lower() for x in ['age', 'interval', 'years', 'count', 'number'] or pd.api.types.is_numeric_dtype(df[c]))
    ]

    # Choose first likely numeric field present
    numeric_field = None
    for c in likely_numeric_fields:
        if pd.api.types.is_numeric_dtype(df[c]) or np.issubdtype(df[c].dtype, np.number):
            numeric_field = c
            break
        # If not, attempt conversion
        try:
            df[c] = pd.to_numeric(df[c], errors='coerce')
            if df[c].notnull().sum() > 0:
                numeric_field = c
                break
        except Exception:
            continue

    if numeric_field:
        print(f"Numeric field selected: {numeric_field}")

        # Set threshold for demonstration (just below max for demonstration if no obvious threshold)
        threshold = df[numeric_field].quantile(0.75) if df[numeric_field].notnull().sum() > 0 else 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (upper quartile):")
        display(filtered_df.head())

        # Normalize
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, normalized_col]].head())

        # Attempt grouping by a categorical field (e.g., sex, location, category, etc.)
        potential_group_fields = [
            c for c in df.columns 
            if df[c].dtype == object and not any(x == c for x in [numeric_field, normalized_col])
        ]
        group_field = None
        for c in potential_group_fields:
            # Use if not too many unique values
            if df[c].nunique() < df.shape[0] / 4:
                group_field = c
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field detected for EDA.\nColumns:", df.columns.tolist())
else:
    print("No dataframe available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using discovered field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only generate plots if there is a main dataframe and numeric field
if dataframes:
    df = list(dataframes.values())[0]
    # Use numeric_field and group_field identified above (recompute if not in this scope)
    # Try best to auto-detect
    numeric_field = None
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_field = c
            break
        try:
            as_num = pd.to_numeric(df[c], errors='coerce')
            if as_num.notnull().sum() > 0:
                numeric_field = c
                df[c] = as_num
                break
        except Exception:
            continue

    group_field = None
    for c in df.columns:
        if df[c].dtype == object and df[c].nunique() < df.shape[0] / 4:
            group_field = c
            break

    if numeric_field:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field].dropna(), kde=True)
        plt.title(f'Distribution of {numeric_field}')
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.tight_layout()
        plt.show()

    if numeric_field and group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.tight_layout()
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
With the `mlcroissant` library, we've explored structured clinical data on second primary colorectal cancers in survivors, starting with schema inspection (by `@id`), custom extraction, EDA (filtering, normalization, grouping), and basic statistical visualization. The dataset's well-structured Croissant metadata makes such analysis reproducible and reusable.

Further advanced analyses can be performed using record set and field `@id` references, leveraging the full richness of the Croissant metadata model.